In [ ]:
"""
    create_dataset-dq_ccda_snooper_section.ipynb
    Creates and transforms/publishes a dataset for DQ.
    
    output: dq_ccda_snooper_section
"""    

In [ ]:
import os  # Provides functions for interacting with the operating system (e.g., file paths, environment variables)
import argparse  # Used for parsing command-line arguments
import re  # Provides support for working with regular expressions
import lxml.etree as ET  # Part of the lxml library for XML parsing and manipulation
from xml_ns import ns  # Imports a custom module or dictionary named 'ns' for handling XML namespaces
import pandas as pd  # Pandas library for handling and analyzing structured data (e.g., CSV, DataFrames)
from collections import defaultdict  # A dictionary subclass that provides default values for missing keys
from foundry.transforms import Dataset # library for working Foundry Datasets

In [ ]:
# Output dataset columns
df_columns = [  
    'filename',         # Name of the file being processed
    'section',          # OID of the section within the document (e.g. 2.16.840.1.113883.10.20.22.2.5.1)
    'section_code',     # Code representing the section (e.g. 11450-4)
    'section_name',     # Human-readable name of the section (e.g. PROBLEM LIST)
    'codeSystem',       # OID system identifier (e.g., 2.16.840.1.113883.6.96 = SNOMED CT)
    'code',             # Specific code within the code system
    'value_type',       # Type of value (e.g., numeric, text, coded entry)
    'value_unit',       # Unit of measurement (if applicable)
    'value_value',      # Actual value extracted
    'value_code',       # Code associated with the value (if applicable)
    'value_codeSystem', # Code system for the value
    'value_text',       # Human-readable text representation of the value
    'path'              # XML path to the extracted data point
]

In [ ]:
def snoop_section(tree, filename:str):
    """
    Extracts structured data from CCDA XML sections and returns a DataFrame.
    Returns: pd.DataFrame
    """

    # Initialize an empty DataFrame with predefined columns
    trace_df = pd.DataFrame(columns=df_columns)

    # Find all 'section' elements in the XML tree using the specified namespace
    section_elements = tree.findall(".//section", ns)

    for section_element in section_elements:

        # Default section template ID if not found
        section_template_id = "n/a"

        # Extract templateId from the section (if available)
        section_template_ele = section_element.findall("templateId", ns)
        if len(section_template_ele) > 0:
            section_template_id = section_template_ele[0].get('root')

        # Extract section code and display name
        section_code = section_element.findall("code", ns)[0].get('code')
        section_name = section_element.findall("code", ns)[0].get('displayName')

        # Find all 'entry' elements within the section
        entry_elements = section_element.findall("entry", ns)

        for entry_ele in entry_elements:

            # Dictionary to store extracted values, grouped by XML path
            value_dict = defaultdict(list)

            # Extract all 'value' elements within the entry
            value_elements = entry_ele.findall(".//value", ns)
            for value_ele in value_elements:

                # Clean the XML path (remove namespace references)
                value_path = re.sub(r'{.*?}', '', tree.getelementpath(value_ele))
                value_path = "/".join(value_path.split("/")[:-1])  # Get parent path

                # Extract attributes from the 'value' element and clean namespaces
                #value_attribs_dict = {
                #    re.sub(r'{.*}', '', a): re.sub(r'{.*}', '', value_ele.attrib[a])
                #    for a in value_ele.attrib
                #}
                value_attribs_dict = {}
                for (attr, value) in value_ele.attrib.items():
                    clean_attr = re.sub(r'{.*}', '', attr)
                    clean_value = re.sub(r'{.*}', '', value)
                    value_attribs_dict[clean_attr] = clean_value

                # Dict of (attribute-dict, text-value) pairs, keyed by path
                value_dict[value_path].append((value_attribs_dict, value_ele.text))
            # Extract all 'code' elements within the entry
            code_elements = entry_ele.findall(".//code", ns)
            for code_ele in code_elements:

                # Clean the XML path (remove namespace references)
                code_path = re.sub(r'{.*?}', '', tree.getelementpath(code_ele))
                code_path = "/".join(code_path.split("/")[:-1])  # Get parent path

                # Retrieve corresponding value(s) for the code (if any)
                value_tuple_list = value_dict[code_path]  # Tuple contains (attributes dictionary, text content)

                for value_tuple in value_tuple_list:
                    # Construct a new row for the DataFrame
                    new_row = pd.DataFrame([{
                        'filename': filename,
                        'section': section_template_id,
                        'section_code': section_code,
                        'section_name': section_name,
                        'path': code_path,
                        'code': code_ele.get('code'),
                        'codeSystem': code_ele.get('codeSystem'),
                        'value_type': value_tuple[0].get('type', ''),  # Extract 'type' if available
                        'value_unit': value_tuple[0].get('unit', ''),  # Extract 'unit' if available
                        'value_value': value_tuple[0].get('value', ''),  # Extract 'value' if available
                        'value_code': value_tuple[0].get('code', ''),  # Extract 'code' if available
                        'value_codeSystem': value_tuple[0].get('codeSystem', ''),  # Extract 'codeSystem' if available
                        'value_text': value_tuple[1].strip() if value_tuple[1] else ''  # Extract and clean text content
                    }])

                    trace_df = pd.concat([trace_df, new_row], ignore_index=True)
    return trace_df

In [ ]:
def main():
    #TODO: Convert to use dataset containing files
    directory = 'resources'
    
    #Process multiple files in a directory
    if directory is not None:
        # Initialize an empty DataFrame to store results from all files
        all_files_df = pd.DataFrame(columns=df_columns)

        # List all files in the given directory, filtering for actual files
        only_files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]

        for filename in only_files:
            if filename.endswith(".xml"):  # Process only XML files
                tree = ET.parse(os.path.join(directory, filename))  # Parse the XML file
                file_df = snoop_section(tree, filename)  # Extract structured data

                # Append the new file's data to the cumulative DataFrame
                all_files_df = pd.concat([all_files_df, file_df], ignore_index=True)

        # Save the results to a CSV file
        #all_files_df.to_csv("raw_section_snooper_new.csv", sep=",", header=True, index=False)
        
        df_dq_ccda_snooper_section = all_files_df
        
        #Save the dataset to Foundry
        dq_ccda_snooper_section = Dataset.get("dq_ccda_snooper_section")
        dq_ccda_snooper_section.write_table(df_dq_ccda_snooper_section)

In [ ]:
#run main
main()